In [ ]:
!pip install --upgrade torchao peft

In [ ]:
# ================================================
#  mBART‑large‑50 + LoRA – Faster Training Script
# ================================================
# 1. Upload small.csv, medium.csv, large.csv to /content/
# 2. Run this cell. Training ~7‑8 hours on T4.
# 3. Download mbart-lora-final.zip from the Files panel.
# ================================================

# ---------- 1. Install packages ----------
!pip install -q transformers datasets pandas scikit-learn sentencepiece peft accelerate

# ---------- 2. Imports ----------
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import torch, shutil, os

# ---------- 3. Tokenizer & Base Model ----------
model_name = "facebook/mbart-large-50"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ---------- 4. Apply LoRA ----------
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.base_model.gradient_checkpointing_enable()
print("✅ mBART‑large‑50 + LoRA loaded.")

# ---------- 5. Load dataset (robust encoding) ----------
def safe_read(path):
    try:
        return pd.read_csv(path, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin-1')

small  = safe_read("/content/small.csv")
medium = safe_read("/content/medium.csv")
large  = safe_read("/content/large.csv")
all_data = pd.concat([small, medium, large], ignore_index=True)
print(f"📊 Total rows: {all_data.shape[0]}")

# ---------- 6. Train/val/test split ----------
train_df, temp_df = train_test_split(all_data, test_size=0.2, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)
print(f"Train: {train_df.shape[0]} | Val: {val_df.shape[0]} | Test: {test_df.shape[0]}")

# ---------- 7. Tokenize ----------
train_dataset = Dataset.from_pandas(train_df[['content', 'summary']])
val_dataset   = Dataset.from_pandas(val_df[['content', 'summary']])
test_dataset  = Dataset.from_pandas(test_df[['content', 'summary']])

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

def tokenize_function(examples):
    model_inputs = tokenizer(
        examples['content'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=examples['summary'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print("🔠 Tokenizing...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val   = val_dataset.map(tokenize_function, batched=True)
tokenized_test  = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(['content', 'summary'])
tokenized_val   = tokenized_val.remove_columns(['content', 'summary'])
tokenized_test  = tokenized_test.remove_columns(['content', 'summary'])
print("✅ Tokenization done.")

# ---------- 8. Training arguments (faster) ----------
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mbart-lora-nepali",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=500,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=1e-4,
    warmup_steps=500,
    weight_decay=0.01,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=128,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    save_total_limit=1,
    max_grad_norm=1.0,
    dataloader_num_workers=4,
)

# ---------- 9. Train ----------
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("🚀 Starting faster training...")
trainer.train()
print("✅ Training finished.")

# ---------- 10. Quick test ----------
print("\n🔍 Generating a test summary...")
sample = test_df.iloc[0]['content']
inputs = tokenizer(sample, return_tensors="pt", max_length=MAX_INPUT_LENGTH, truncation=True).to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_length=128, num_beams=4, early_stopping=True)
test_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("   Article (truncated):", sample[:200])
print("   Generated summary :", test_summary)

# ---------- 11. Save final model ----------
print("\n💾 Saving final model...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./mbart-lora-final")
tokenizer.save_pretrained("./mbart-lora-final")
print("   Final model saved to ./mbart-lora-final")

# ---------- 12. Create ZIP ----------
print("📦 Creating mbart-lora-final.zip...")
shutil.make_archive("/content/mbart-lora-final", 'zip', "./mbart-lora-final")
print("   ZIP created: /content/mbart-lora-final.zip")

# ---------- 13. Cleanup ----------
shutil.rmtree("./mbart-lora-nepali", ignore_errors=True)
print("   Cleaned up checkpoints.")

print("\n🎉 ALL DONE! Download mbart-lora-final.zip from the Files panel.")

Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

trainable params: 1,179,648 || all params: 612,059,136 || trainable%: 0.1927
✅ mBART‑large‑50 + LoRA loaded.
📊 Total rows: 50529
Train: 40423 | Val: 5053 | Test: 5053
🔠 Tokenizing...


Map:   0%|          | 0/40423 [00:00<?, ? examples/s]

Map:   0%|          | 0/5053 [00:00<?, ? examples/s]

Map:   0%|          | 0/5053 [00:00<?, ? examples/s]

✅ Tokenization done.
🚀 Starting faster training...


... (2 outputs left out)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/to

✅ Training finished.

🔍 Generating a test summary...
   Article (truncated): १८ असोज, काठमाडौं। स्वास्थ्य बीमा बोर्डले पञ्चवर्षीय रणनीतिक मार्गचित्र सार्वजनिक गरेको छ । बोर्डले ९ वटा रणनीति तय गरेको छ । संरचनागत सुदृढीकरण तथा सुशासनलाई बोर्डले पहिलो प्राथमिकतामा राखेको छ। यसैग
   Generated summary : नेपाल स्वास्थ्य बीमा बोर्डले पञ्चवर्षीय रणनीतिक मार्गचित्र सार्वजनिक गरेको छ, जसमा संरचनागत सुदृढीकरण र सुशासनलाई प्राथमिकता दिइएको छ। दिगो वित्तीय व्यवस्थापन, बीमा व्यवस्थापन सूचना प्रणालीको सुदृढीकरण, दाबी व्यवस्थापन प्रणालीको परिमार्जन, अनुगमन तथा मूल्यांकन प्रणालीमा सुधार र कार्यक्रमको प्रभावकारी कार्यान्वयनका लागि समन्वय र सहकार्य आवश्यक रहेको छ। स्वास्थ्य बीमाको वित्तीय दिगोपना र नागरिकको आवद्धतामा वृद्धिका कारण गुणस्तरीय स्वास्थ्य सेवाप्रति नागरिकको असन्तुष्टि बढेको छ।

💾 Saving final model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   Final model saved to ./mbart-lora-final
📦 Creating mbart-lora-final.zip...
   ZIP created: /content/mbart-lora-final.zip
   Cleaned up checkpoints.

🎉 ALL DONE! Download mbart-lora-final.zip from the Files panel.


In [ ]:
import os

print("Checking files...")

print("Folder exists:",
      os.path.exists("/content/mbart-lora-final"))

print("ZIP exists:",
      os.path.exists("/content/mbart-lora-final.zip"))

if os.path.exists("/content/mbart-lora-final"):
    print("\nFiles inside model folder:")
    print(os.listdir("/content/mbart-lora-final")[:20])

Checking files...
Folder exists: False
ZIP exists: False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')